# Checkpoint 3 — Contest 3 Practice Solutions

Worked answers for Questions 1–8, and a reference program for Question 9.
Each check cell works its answer out again with a few lines of Python: a full truth table, a queue run, a tree build, the automaton's table, or a comparison of two expressions on every short string.
The two program questions are copied into the `assets` folder (`assets/trace5.py` and `assets/trace6.py`) and run.
The Question 9 program is saved as `assets/q9.py`; run it with `python assets/q9.py < input.txt`.
If you missed a question, go back to its unit: Questions 1–2 are Boolean Algebra, 3–4 are Data Structures, 5–6 are What Does This Program Do? – Arrays, and 7–8 are FSAs and Regular Expressions.

## Question 1

Rewrite the last term with De Morgan, then combine terms.

- `~(A + ~C)` = `~A * ~~C` = `~A * C`.
- `A * B + A * ~B * C` = `A * (B + ~B * C)` = `A * (B + C)` = `A * B + A * C`.
- Now the expression is `A * B + A * C + ~A * C`, and `A * C + ~A * C` = `(A + ~A) * C` = `C`.
- That leaves `A * B + C`.

Check with the truth table: the expression is true exactly when `C` is 1 or both `A` and `B` are 1 (rows `001, 011, 101, 110, 111`).
Neither term can be dropped or shortened, so `A * B + C` is the only simplest form.

**Answer:** `A * B + C`

In [ ]:
from itertools import combinations, product

def tokens(text):
    out = []
    for ch in text:
        if ch != " ":
            out.append(ch)
    return out

def parse(toks, pos, level):
    # level 0: +, level 1: ⊕ and ⊙, level 2: *, level 3: ~ and atoms
    if level == 3:
        t = toks[pos]
        if t == "~":
            f, pos = parse(toks, pos + 1, 3)
            return (lambda v, f=f: 1 - f(v)), pos
        if t == "(":
            f, pos = parse(toks, pos + 1, 0)
            return f, pos + 1
        if t in "01":
            return (lambda v, t=t: int(t)), pos + 1
        return (lambda v, t=t: v[t]), pos + 1
    ops = {0: {"+": lambda x, y: x | y},
           1: {"⊕": lambda x, y: x ^ y, "⊙": lambda x, y: 1 - (x ^ y)},
           2: {"*": lambda x, y: x & y}}[level]
    f, pos = parse(toks, pos, level + 1)
    while pos < len(toks) and toks[pos] in ops:
        op = ops[toks[pos]]
        g, pos = parse(toks, pos + 1, level + 1)
        f = (lambda v, f=f, g=g, op=op: op(f(v), g(v)))
    return f, pos

def compile_expr(text):
    toks = tokens(text)
    f, pos = parse(toks, 0, 0)
    assert pos == len(toks)
    names = sorted(set(t for t in toks if t.isalpha()))
    return f, names

def rows(names):
    return [dict(zip(names, bits)) for bits in product([0, 1], repeat=len(names))]

expr = "A * B + A * ~B * C + ~(A + ~C)"
f, names = compile_expr(expr)
ones = [tuple(r[n] for n in names) for r in rows(names) if f(r) == 1]

# every product term: each variable is left out (None), plain (1) or negated (0)
terms = [t for t in product([None, 1, 0], repeat=len(names)) if any(x is not None for x in t)]

def covers(t):
    return set(r for r in product([0, 1], repeat=len(names))
               if all(x is None or r[i] == x for i, x in enumerate(t)))

implicants = [t for t in terms if covers(t) <= set(ones)]
best = None
found = []
for k in range(1, len(implicants) + 1):
    for combo in combinations(implicants, k):
        if set().union(*[covers(t) for t in combo]) == set(ones):
            cost = (k, sum(x is not None for t in combo for x in t))
            if best is None or cost < best:
                best, found = cost, [combo]
            elif cost == best:
                found.append(combo)
    if found:
        break

def render(combo):
    keyed = []
    for t in combo:
        lits = [(names[i], 1 - x) for i, x in enumerate(t) if x is not None]
        keyed.append(lits)
    keyed.sort()
    parts = []
    for lits in keyed:
        parts.append(" * ".join(("~" if neg else "") + v for v, neg in lits))
    return " + ".join(parts)

minimal = sorted(set(render(c) for c in found))
assert len(minimal) == 1
assert str(minimal[0]) == "A * B + C"

## Question 2

Use the precedence `~`, then `*`, then `⊕`, then `+`.
So the expression is `(~(A * B) ⊕ C) + (A * ~B * ~C)`.

| A B C | ~(A * B) | ~(A * B) ⊕ C | A * ~B * ~C | result |
|---|---|---|---|---|
| 0 0 0 | 1 | 1 | 0 | 1 |
| 0 0 1 | 1 | 0 | 0 | 0 |
| 0 1 0 | 1 | 1 | 0 | 1 |
| 0 1 1 | 1 | 0 | 0 | 0 |
| 1 0 0 | 1 | 1 | 1 | 1 |
| 1 0 1 | 1 | 0 | 0 | 0 |
| 1 1 0 | 0 | 0 | 0 | 0 |
| 1 1 1 | 0 | 1 | 0 | 1 |

Four rows are true.

**Answer:** `(0,0,0), (0,1,0), (1,0,0), (1,1,1)`

In [ ]:
from itertools import combinations, product

def tokens(text):
    out = []
    for ch in text:
        if ch != " ":
            out.append(ch)
    return out

def parse(toks, pos, level):
    # level 0: +, level 1: ⊕ and ⊙, level 2: *, level 3: ~ and atoms
    if level == 3:
        t = toks[pos]
        if t == "~":
            f, pos = parse(toks, pos + 1, 3)
            return (lambda v, f=f: 1 - f(v)), pos
        if t == "(":
            f, pos = parse(toks, pos + 1, 0)
            return f, pos + 1
        if t in "01":
            return (lambda v, t=t: int(t)), pos + 1
        return (lambda v, t=t: v[t]), pos + 1
    ops = {0: {"+": lambda x, y: x | y},
           1: {"⊕": lambda x, y: x ^ y, "⊙": lambda x, y: 1 - (x ^ y)},
           2: {"*": lambda x, y: x & y}}[level]
    f, pos = parse(toks, pos, level + 1)
    while pos < len(toks) and toks[pos] in ops:
        op = ops[toks[pos]]
        g, pos = parse(toks, pos + 1, level + 1)
        f = (lambda v, f=f, g=g, op=op: op(f(v), g(v)))
    return f, pos

def compile_expr(text):
    toks = tokens(text)
    f, pos = parse(toks, 0, 0)
    assert pos == len(toks)
    names = sorted(set(t for t in toks if t.isalpha()))
    return f, names

def rows(names):
    return [dict(zip(names, bits)) for bits in product([0, 1], repeat=len(names))]

expr = "~(A * B) ⊕ C + A * ~(B + C)"
f, names = compile_expr(expr)
true_rows = []
for r in rows(names):
    if f(r) == 1:
        true_rows.append("(" + ",".join(str(r[n]) for n in names) + ")")
assert str(", ".join(true_rows)) == "(0,0,0), (0,1,0), (1,0,0), (1,1,1)"

## Question 3

A queue is first in, first out: `POP` takes the item at the front.

| Statement | Queue (front first) | Variable |
|---|---|---|
| `PUSH(6)`, `PUSH(1)`, `PUSH(9)` | 6 1 9 | |
| `X = POP()` | 1 9 | X = 6 |
| `PUSH(4)` | 1 9 4 | |
| `Y = POP()` | 9 4 | Y = 1 |
| `PUSH(X)`, `PUSH(Y)` | 9 4 6 1 | |
| `Z = POP()` | 4 6 1 | Z = 9 |
| `Z = POP()` | 6 1 | Z = 4 |

**Answer:** `4`

In [ ]:
script = ["PUSH(6)", "PUSH(1)", "PUSH(9)", "X = POP()", "PUSH(4)", "Y = POP()",
          "PUSH(X)", "PUSH(Y)", "Z = POP()", "Z = POP()"]
queue = []
variables = {}
for line in script:
    if line.startswith("PUSH("):
        item = line[5:-1]
        queue.append(variables[item] if item in variables else int(item))
    else:
        name = line.split(" = ")[0]
        variables[name] = queue.pop(0) if queue else "NIL"
assert str(variables["Z"]) == "4"

## Question 4

Insert `S E P T E M B E R` in order; a letter equal to a node's letter goes left, and the root has depth 0.

| Letter | Path from the root | Depth |
|---|---|---|
| S | root | 0 |
| E | left of S | 1 |
| P | S left, E right | 2 |
| T | right of S | 1 |
| E | S left, E left (equal goes left) | 2 |
| M | S left, E right, P left | 3 |
| B | S left, E left, E left | 3 |
| E | S left, E left, E left, B right | 4 |
| R | S left, E right, P right | 3 |

The internal path length is 0 + 1 + 2 + 1 + 2 + 3 + 3 + 4 + 3 = 19.

**Answer:** `19`

In [ ]:
word = "SEPTEMBER"
keys, left, right, depths = [], [], [], []
for letter in word:
    keys.append(letter)
    left.append(-1)
    right.append(-1)
    new = len(keys) - 1
    d = 0
    node = 0
    while new > 0:
        d += 1
        side = left if letter <= keys[node] else right
        if side[node] == -1:
            side[node] = new
            break
        node = side[node]
    depths.append(d)
assert str(sum(depths)) == "19"

## Question 5

`n` is 7, so the first loop runs for `i` = 0, 1, 2 and swaps each item with its mirror: the list is reversed to `[2, 6, 3, 8, 1, 7, 4]`.

The second loop goes left to right, and whenever an item is smaller than the (already updated) item before it, it adds that item on.

| i | A[i] | A[i - 1] | smaller? | new A[i] |
|---|---|---|---|---|
| 1 | 6 | 2 | no | 6 |
| 2 | 3 | 6 | yes | 9 |
| 3 | 8 | 9 | yes | 17 |
| 4 | 1 | 17 | yes | 18 |
| 5 | 7 | 18 | yes | 25 |
| 6 | 4 | 25 | yes | 29 |

The list ends as `[2, 6, 9, 17, 18, 25, 29]`.
`A[n - 1] - A[0]` is 29 - 2 = 27, and `A[3]` is 17; `print` puts one space between them.

**Answer:** `27 17`

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "assets/trace5.py"], input='', capture_output=True, text=True)
output = result.stdout.rstrip("\n")
assert str(output) == '27 17'

## Question 6

`A(r, c)` = `(r * c + r) % 5`.
The last loop adds the main-diagonal item `A(k, k)` and takes away the anti-diagonal item `A(k, 5 - k)`.

| k | A(k, k) | A(k, 5 - k) |
|---|---|---|
| 1 | (1 + 1) % 5 = 2 | A(1, 4) = (4 + 1) % 5 = 0 |
| 2 | (4 + 2) % 5 = 1 | A(2, 3) = (6 + 2) % 5 = 3 |
| 3 | (9 + 3) % 5 = 2 | A(3, 2) = (6 + 3) % 5 = 4 |
| 4 | (16 + 4) % 5 = 0 | A(4, 1) = (4 + 4) % 5 = 3 |

`s` = (2 + 1 + 2 + 0) - (0 + 3 + 4 + 3) = 5 - 10 = -5.

**Answer:** `-5`

The check cell runs a Python copy of the program (`assets/trace6.py`): each `FOR ... TO 4` loop becomes `range(1, 5)`, and the array gets an unused row 0 and column 0 so that `A[r][c]` matches `A(r, c)`.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "assets/trace6.py"], input='', capture_output=True, text=True)
output = result.stdout.rstrip("\n")
assert str(output) == '-5'

## Question 7

Look at the table: every `a` switches between the pairs `S0`/`S1` and `S2`/`S3`, and every `b` switches between `S0`/`S2` and `S1`/`S3`.
So the automaton only keeps track of whether it has read an odd or even number of `a`s and of `b`s, and `S3` means an odd number of both.

| Option | String | a's | b's | Accepted? |
|---|---|---|---|---|
| A | `ab` | 1 | 1 | yes (S0, S1, S3) |
| B | `aabb` | 2 | 2 | no (ends in S0) |
| C | `ababab` | 3 | 3 | yes |
| D | `bbbab` | 1 | 4 | no (ends in S1) |
| E | `baaab` | 3 | 2 | no (ends in S1) |

**Answer:** `A, C`

In [ ]:
table = {
    ("S0", "a"): "S1", ("S0", "b"): "S2",
    ("S1", "a"): "S0", ("S1", "b"): "S3",
    ("S2", "a"): "S3", ("S2", "b"): "S0",
    ("S3", "a"): "S2", ("S3", "b"): "S1",
}
options = {"A": "ab", "B": "aabb", "C": "ababab", "D": "bbbab", "E": "baaab"}

def accepts(s):
    state = "S0"
    for symbol in s:
        if (state, symbol) not in table:
            return False
        state = table[(state, symbol)]
    return state == "S3"

chosen = [label for label in sorted(options) if accepts(options[label])]
assert str(", ".join(chosen) or "NONE") == "A, C"

## Question 8

`a(ba)*` describes `a`, `aba`, `ababa`, …: strings that start and end with `a` and alternate letters.

- A. `(ab)*a` gives `a`, `aba`, `ababa`, …: the same strings, grouped the other way. Equivalent.
- B. `a(ab)*` gives `aab`, which is not in the set. Not equivalent.
- C. `(ab U λ)*a`: repeating "`ab` or nothing" gives the same strings as `(ab)*`, so this is option A again. Equivalent.
- D. `a U a(ba)*ba`: `a(ba)*ba` is `a` followed by one or more `ba`, and adding `a` itself gives zero or more. Equivalent.
- E. `ab*a*` gives `ab`, which is not in the set. Not equivalent.

**Answer:** `A, C, D`

In [ ]:
import re
from itertools import product

def to_python(pattern):
    # U is union and λ the empty string; none of these patterns has a [...] class
    return pattern.replace(" ", "").replace("U", "|").replace("λ", "()")

def same_language(p, q, alphabet="ab", max_len=8):
    rp, rq = re.compile(to_python(p)), re.compile(to_python(q))
    for n in range(max_len + 1):
        for letters in product(alphabet, repeat=n):
            s = "".join(letters)
            if bool(rp.fullmatch(s)) != bool(rq.fullmatch(s)):
                return False
    return True

target = "a(ba)*"
options = {"A": "(ab)*a", "B": "a(ab)*", "C": "(ab U λ)*a", "D": "a U a(ba)*ba", "E": "ab*a*"}
chosen = [label for label in sorted(options) if same_language(target, options[label])]
assert str(", ".join(chosen) or "NONE") == "A, C, D"

## Question 9

**Notice:** use the BST idiom from Unit 9: parallel lists `key`, `left` and `right`, with `-1` for "no child".
Insert each letter with a `while` loop from the root, going left when the letter is less than or equal to the node's letter, and record the depth (the number of steps taken) as the letter is placed.
Then one pass over the nodes adds up the depths, counts the nodes with no children, and keeps the largest depth.
The fixtures include a one-letter word (`Q` gives `0 1 0`), all letters equal (`AAAA` makes a chain down the left, `6 1 3`), a duplicate that must go left (`BAB` gives `3 1 2`, not `2 2 1`), a sorted word that makes a chain down the right, a perfectly balanced tree of 15 letters (`34 8 3`), `MISSISSIPPI`, and 30-letter words.

In [ ]:
word = input().split()[0]
key = []
left = []
right = []
depth = []
for letter in word:
    key.append(letter)
    left.append(-1)
    right.append(-1)
    new = len(key) - 1
    d = 0
    if new > 0:
        i = 0
        placed = False
        while not placed:
            d = d + 1
            if letter <= key[i]:
                if left[i] == -1:
                    left[i] = new
                    placed = True
                else:
                    i = left[i]
            else:
                if right[i] == -1:
                    right[i] = new
                    placed = True
                else:
                    i = right[i]
    depth.append(d)
total = 0
leaves = 0
deepest = 0
for n in range(len(key)):
    total = total + depth[n]
    if left[n] == -1 and right[n] == -1:
        leaves = leaves + 1
    if depth[n] > deepest:
        deepest = depth[n]
print(str(total) + " " + str(leaves) + " " + str(deepest))